# Javtiful → GoFile Telegram Bot

Minimal Google Colab runner for the `gofile` branch. It uses only two Colab Secrets: `BOT_TOKEN` and `OWNER_ID`. No database, Neon, `API_ID`, `API_HASH`, or GoFile token is needed.

### Run steps
1. Open **Secrets** (key icon) and add `BOT_TOKEN` and `OWNER_ID`; enable notebook access.
2. Run **Setup** once to install the small dependency set and clone the `gofile` branch.
3. Run **Check** to compile the bot and execute its basic tests.
4. Run **Start bot** and send a video code such as `HNTRZ-010` to your Telegram bot.

The bot is owner-only and keeps search sessions in memory. Live Javtiful playback/search behavior can change, so a successful source check does not guarantee the website integration will work.

In [ ]:
# Setup — load secrets, install ffmpeg, and fetch only the gofile branch
import os
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

for key in ("BOT_TOKEN", "OWNER_ID"):
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if value is None or not str(value).strip():
        raise RuntimeError(f"Missing Colab Secret {key}. Add it in Secrets and enable notebook access.")
    os.environ[key] = str(value).strip()

try:
    owner_id = int(os.environ["OWNER_ID"])
    if owner_id <= 0:
        raise ValueError
except ValueError as exc:
    raise ValueError("OWNER_ID must be your positive numeric Telegram user ID.") from exc

REPO_DIR = Path("/content/URL-uploader-gofile")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "gofile", "https://github.com/Telebotfaroff/URL-uploader.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Setup complete. Secret values were not displayed.")

In [ ]:
# Check — compile the required code and run the GoFile regression tests
import subprocess
import sys

subprocess.run([sys.executable, "-m", "compileall", "-q", "bot.py", "gofile_bot.py", "gofile_search.py", "gofile_media.py", "gofile_uploader.py", "plugins/javdl_core"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_gofile_bot.py", "-v"], check=True)
print("Checks passed.")

In [ ]:
# Start bot — keep this cell running while you use the bot
import os
import subprocess
import sys

if not os.environ.get("BOT_TOKEN") or not os.environ.get("OWNER_ID"):
    raise RuntimeError("Run the Setup cell first so Colab Secrets are loaded.")
subprocess.run([sys.executable, "gofile_bot.py"], check=True)